# Task 1 - Article Type Classification

This notebook asks a simple question: which safe image-only method works best for the 124 article types? It starts with the data problem, then tests clear hypotheses. The default Run All path performs smoke work only, not final training.


## 1. Problem and output

Input: one fashion image for one product `id`. Output: one label from the fixed 124-class `articleType` vocabulary. The final submission needs `id,gender,articleType,season,usage`, so this task fills only `articleType`.

We compare labelled development images only. The learned CNN starts from scratch. Pretrained weights are not part of the submitted model.


## 2. EDA evidence

The prepared evidence below describes image shape, grayscale files, class imbalance, and fold warnings. These are problems to solve. The choices in the table are hypotheses to test, not declared winners.


In [ ]:
from dataclasses import asdict

import pandas as pd

from fashion.config import DEVELOPMENT_CLASS_SUMMARY_CSV
from fashion.data.dataset import get_samples, load_label_maps, load_splits
from fashion.task1 import (
    build_task1_decision_evidence,
    build_task1_problem_profile,
)

TARGET = "articleType"
splits = load_splits()
task1_development = get_samples(splits, partition="development", target=TARGET)
article_type_map = load_label_maps()[TARGET]
article_type_classes = tuple(article_type_map["classes"])
class_summary = pd.read_csv(DEVELOPMENT_CLASS_SUMMARY_CSV, keep_default_na=False)
problem_profile = build_task1_problem_profile(splits, class_summary)

display(pd.DataFrame([asdict(problem_profile)]))
display(build_task1_decision_evidence(problem_profile))


## 3. Safety contract

`data/processed/splits.csv` is the only split. We do not make a new split. Development rows use the saved five folds. Holdout and quarantine labels stay sealed until Notebook 06.

Only image pixels are model input. Names, years, file size, and the other targets are not used because they can be shortcuts. Every physical training run is registered in `results/runs.csv`.


## 4. Evaluation

Each candidate uses all five saved folds. The main score is fixed-label macro-F1 across all 124 classes. Macro-F1 gives a rare class the same weight as a common class, so accuracy cannot hide rare-class failure.

We compare five-fold mean macro-F1 and its sample standard deviation. Per-class F1, out-of-fold predictions, confusion pairs, weighted F1, and accuracy explain the score; they do not replace the fixed decision rule.


## 5. Candidate hypotheses

The experiment story has eight parts: Classical HOG baselines; scratch CNN without augmentation; learning-curve diagnosis; mild augmentation; gentle class-weighted loss; combined five-fold and pooled OOF comparison; rare-class and confusion analysis; then the development decision and Notebook 06 handoff.

The known augmentation result is: lower mean validation loss and lower fold variance, but current mean macro-F1 changed from 0.5291 to 0.5218 and therefore did not pass the macro-F1 improvement rule.

The gentle class-weighted candidate is not judged here until its five full weighted folds exist. Weighted smoke is only a path check.


## 6. Controlled preprocessing

The EDA suggests a shared image contract: EXIF orientation, RGB conversion, shape-preserving 60-by-80 white padding, then fold-fitted normalization. This stops stretching and handles grayscale images the same way for every family.

The control has no random change. The augmentation condition adds small seeded flips, rotations, movement, scale, brightness, and contrast changes. The weighted-loss condition returns to the no-augmentation images so the loss change is isolated.


In [ ]:
from fashion.task1 import (
    DEFAULT_TASK1_PREPROCESSING,
    TASK1_CONTROL_PREPROCESSING,
    TASK1_GENTLE_WEIGHTED_CANDIDATE,
    TASK1_MILD_AUG_CANDIDATE,
    TASK1_NO_AUG_CANDIDATE,
    run_task1_classical_experiment,
    run_task1_experiment,
)

preprocessing_candidates = {
    "control_no_augmentation": TASK1_CONTROL_PREPROCESSING.to_dict(),
    "hypothesis_mild_augmentation": DEFAULT_TASK1_PREPROCESSING.to_dict(),
}
cnn_candidates = pd.DataFrame(
    [
        {
            "candidate_id": candidate.candidate_id,
            "preprocessing_id": candidate.preprocessing.preprocessing_id,
            "loss_id": candidate.loss.loss_id,
        }
        for candidate in (
            TASK1_NO_AUG_CANDIDATE,
            TASK1_MILD_AUG_CANDIDATE,
            TASK1_GENTLE_WEIGHTED_CANDIDATE,
        )
    ]
)

display(pd.DataFrame(preprocessing_candidates).T)
display(cnn_candidates)


## 7. Classical baselines and scratch CNN controllers

First, the HOG baselines give a non-neural comparison. Then the scratch CNN controller checks the no-augmentation and mild-augmentation image learners. Both controllers default to smoke mode, so Run All stays quick and does not create five-fold evidence by accident.

Change `CLASSICAL_STAGE` or `RUN_MODE` only when you mean to run the longer registered stages.


In [ ]:
CLASSICAL_STAGE = "smoke"  # Use "tune" and then "final" only as separate deliberate runs.
classic_experiment = run_task1_classical_experiment(
    splits,
    article_type_map,
    stage=CLASSICAL_STAGE,
)
display(classic_experiment.fold_metrics)
if not classic_experiment.tuning.empty:
    display(classic_experiment.tuning)
if not classic_experiment.comparison.empty:
    display(classic_experiment.comparison)
if not classic_experiment.oof_metrics.empty:
    display(classic_experiment.oof_metrics)


In [ ]:
RUN_MODE = "smoke"  # Use "full" only for the ten registered unweighted CNN folds.
task1_experiment = run_task1_experiment(
    splits,
    article_type_map,
    mode=RUN_MODE,
)
display(task1_experiment.fold_metrics)
if not task1_experiment.comparison.empty:
    display(task1_experiment.comparison)
if not task1_experiment.oof_metrics.empty:
    display(task1_experiment.oof_metrics)


## 8. Learning-curve diagnosis

The no-augmentation scratch CNN showed overfitting risk: validation loss did not improve as cleanly as training loss. Mild augmentation was tested because it should reduce that gap. It did reduce mean validation loss and fold variance, but it did not improve the current mean macro-F1.

The figure below is written only after the combined 15-fold CNN evidence exists. Until then, it asks for the weighted full run instead of launching training.


In [ ]:
from collections import defaultdict
from pathlib import Path

from fashion.config import ROOT, TASK1_EVIDENCE_DIR, TASK1_FIGURE_DIR
from fashion.task1 import write_task1_learning_curve_figure
from fashion.train.registry import RunRegistry

expected_learning_candidates = {
    TASK1_NO_AUG_CANDIDATE.candidate_id,
    TASK1_MILD_AUG_CANDIDATE.candidate_id,
    TASK1_GENTLE_WEIGHTED_CANDIDATE.candidate_id,
}
fold_metrics_path = TASK1_EVIDENCE_DIR / "fold_metrics.csv"
learning_curve_histories = defaultdict(list)

if not fold_metrics_path.is_file():
    print("Learning curves are not ready; complete weighted full first.")
else:
    combined_folds = pd.read_csv(fold_metrics_path, keep_default_na=False)
    required_columns = {"run_id", "candidate_id", "fold"}
    has_required_columns = required_columns.issubset(combined_folds.columns)
    has_complete_folds = False
    if has_required_columns:
        fold_counts = combined_folds.groupby("candidate_id")["fold"].nunique()
        has_complete_folds = (
            len(combined_folds) == 15
            and set(combined_folds["candidate_id"]) == expected_learning_candidates
            and set(fold_counts) == {5}
        )

    if not has_required_columns or not has_complete_folds:
        print("Learning curves are not ready; complete weighted full first.")
    else:
        registry_rows = RunRegistry().read().set_index("run_id", drop=False)
        missing_history = []
        for row in combined_folds.sort_values(["candidate_id", "fold"]).itertuples(index=False):
            if row.run_id not in registry_rows.index:
                missing_history.append(str(row.run_id))
                continue
            registry_row = registry_rows.loc[row.run_id]
            history_path = Path(str(registry_row["history_path"]))
            if not history_path.is_absolute():
                history_path = ROOT / history_path
            if not history_path.is_file():
                missing_history.append(str(row.run_id))
                continue
            learning_curve_histories[str(row.candidate_id)].append(pd.read_csv(history_path))

        if missing_history or any(len(items) != 5 for items in learning_curve_histories.values()):
            print("Learning curves are not ready; complete weighted full first.")
        else:
            learning_curve_path = write_task1_learning_curve_figure(
                dict(learning_curve_histories),
                output=TASK1_FIGURE_DIR / "cnn_learning_curves.png",
            )
            print(f"Wrote {learning_curve_path}")


## 9. Gentle class-weighted loss

The new test keeps the no-augmentation scratch CNN setup and changes only the training loss. Class weights come from each fold's development-training rows only. Validation loss stays unweighted, so the three CNN candidates remain comparable.

Default mode is smoke. Use full only when ready to run the five new weighted folds.


In [ ]:
from fashion.task1 import run_task1_weighted_experiment

WEIGHTED_MODE = "smoke"  # Use "full" only for the five new weighted folds.
weighted_experiment = run_task1_weighted_experiment(
    splits,
    article_type_map,
    mode=WEIGHTED_MODE,
)
display(weighted_experiment.fold_metrics)
if not weighted_experiment.comparison.empty:
    display(weighted_experiment.comparison)
    display(weighted_experiment.oof_metrics)


## 10. Combined five-fold and OOF comparison

This section reads the current Task 1 evidence files. Before weighted full exists, the files may still show only the two old unweighted CNN candidates. After weighted full completes, the same display should show three CNN candidates and 15 fold rows.

Scores come from registered evidence files. Do not hand-write weighted scores in markdown.


In [ ]:
from fashion.config import TASK1_EVIDENCE_DIR

for evidence_name in ("fold_metrics", "comparison", "oof_metrics"):
    evidence_path = TASK1_EVIDENCE_DIR / f"{evidence_name}.csv"
    if evidence_path.is_file():
        print(evidence_name)
        display(pd.read_csv(evidence_path, keep_default_na=False))
    else:
        print(f"{evidence_name} is not ready; complete the needed full runs first.")


In [ ]:
from fashion.config import TASK1_FIGURE_DIR
from fashion.task1 import write_task1_comparison_figure, write_task1_confusion_figure

if WEIGHTED_MODE == "full" and not weighted_experiment.fold_metrics.empty:
    write_task1_comparison_figure(weighted_experiment.fold_metrics)
    for candidate_id, predictions in weighted_experiment.oof_predictions.items():
        write_task1_confusion_figure(
            predictions,
            article_type_classes,
            output=TASK1_FIGURE_DIR / f"cnn_oof_confusion_{candidate_id}.png",
        )
elif RUN_MODE == "full" and not task1_experiment.fold_metrics.empty:
    write_task1_comparison_figure(task1_experiment.fold_metrics)
    for candidate_id, predictions in task1_experiment.oof_predictions.items():
        write_task1_confusion_figure(
            predictions,
            article_type_classes,
            output=TASK1_FIGURE_DIR / f"cnn_oof_confusion_{candidate_id}.png",
        )

if CLASSICAL_STAGE == "final":
    for candidate_id, predictions in classic_experiment.oof_predictions.items():
        write_task1_confusion_figure(
            predictions,
            article_type_classes,
            output=TASK1_FIGURE_DIR / f"classical_oof_confusion_{candidate_id}.png",
        )


## 11. Weak-class/confusion analysis

The tables below use in-memory evidence returned by completed controllers. `example_ids` point to prepared development rows for representative-image inspection. They help find failures, but never choose or rank a model alone.


In [ ]:
from fashion.task1 import (
    build_task1_confusion_pairs,
    build_task1_weak_class_table,
)

per_class_evidence = {
    **{f"cnn:{key}": value for key, value in task1_experiment.per_class.items()},
    **{f"weighted:{key}": value for key, value in weighted_experiment.per_class.items()},
    **{f"classic:{key}": value for key, value in classic_experiment.per_class.items()},
}
oof_prediction_evidence = {
    **{f"cnn:{key}": value for key, value in task1_experiment.oof_predictions.items()},
    **{f"weighted:{key}": value for key, value in weighted_experiment.oof_predictions.items()},
    **{f"classic:{key}": value for key, value in classic_experiment.oof_predictions.items()},
}

if per_class_evidence:
    display(build_task1_weak_class_table(per_class_evidence, limit=10))
else:
    print("Weak-class evidence is not ready; complete full/final runs first.")

if oof_prediction_evidence:
    display(build_task1_confusion_pairs(oof_prediction_evidence, limit=10))
else:
    print("Confusion-pair evidence is not ready; complete full/final runs first.")


## 12. Development decision and Notebook 06 handoff

The augmentation hypothesis did not pass the macro-F1 improvement rule: current mean macro-F1 changed from 0.5291 to 0.5218, even though mean validation loss and fold variance were lower.

The weighted candidate is not ready for judgement until the weighted full stage completes and the combined 15-fold evidence is written. After that, mark the hypothesis passed or failed by five-fold mean macro-F1 first, then fold standard deviation, pooled OOF macro-F1, rare-class F1, common-class impact, learning curves, runtime, and model size.

**NOT READY.** Before Notebook 06, complete the weighted `full` stage, confirm the combined evidence has three CNN candidates, and record the selected run ID, preprocessing configuration, loss ID, fixed metric, five-fold evidence, refit steps, output path, and honest limits. Do not use holdout labels to make this development decision.
